In [4]:
import os, sys, importlib
sys.path.append(os.path.abspath(".."))
import json
import pandas as pd

In [5]:
path_df="../data_raw\listings_paris2306.csv"
df=pd.read_csv(path_df)
print(df.shape)
# display(df.head())

(61706, 75)


In [9]:
path_df_nextQ="../data_raw\listings_paris2309.csv"
df_nextQ=pd.read_csv(path_df_nextQ)
print(df_nextQ.shape)

(67942, 75)


In [ ]:
from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars


df_processed=preprocess_host_variables(dfQ2)
df_filtered=preprocess_obj_vars(df=df_processed, 
            df_nextQ=df_nextQ, 
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            output_folder="../data_processed_paris2306", 
            filename=f"listings_paris2306_filtered.csv"#***
)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, property_type, minimum_night